In [19]:
# 1_EDA_data_prep.ipynb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import joblib

BASE_DIR = r"C:\Users\User-NB\OneDrive\Desktop\ML"
RAW_CSV = os.path.join(BASE_DIR, "Fundamentals Annual_withY.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "processed_data")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading Data...")
df = pd.read_csv(RAW_CSV)
if 'y' in df.columns: df.rename(columns={'y': 'Y'}, inplace=True)

# 1. 定義特徵 (22個原始變數)
exclude_cols = ['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'gvkey', 'datadate', 'cik', 'sic', 'fyear', 'Y', 'tic', 'conm', 'SIC2']
all_features = [c for c in df.columns if c not in exclude_cols]
print(f"Features detected: {len(all_features)}")

# 2. 統一進行高級資料清洗 (Advanced Preprocessing)
print("Processing Data (SIC2 Imputation + Log + Winsorize)...")
work = df.copy()

# A. SIC2 Grouped Imputation
work['sic'] = work['sic'].fillna(0).astype(int).astype(str).str.zfill(4)
work['SIC2'] = work['sic'].str[:2]
train_mask = (work['fyear'] >= 2019) & (work['fyear'] <= 2023)

for col in all_features:
    group_medians = work[train_mask].groupby(['SIC2', 'fyear'])[col].median()
    global_median = work.loc[train_mask, col].median()
    mapped = work.set_index(['SIC2', 'fyear']).index.map(group_medians)
    work[col] = work[col].fillna(pd.Series(mapped, index=work.index)).fillna(global_median)

# B. Signed Log Transform
def sign_log1p(x): return np.sign(x) * np.log1p(np.abs(x))
for col in all_features:
    work[col] = sign_log1p(work[col])

# C. Winsorization (1%-99%)
for col in all_features:
    lo, hi = work[col].quantile([0.01, 0.99])
    work[col] = work[col].clip(lo, hi)

# 3. 標準化
scaler = StandardScaler()
X_processed = scaler.fit_transform(work[all_features].values)
y = work["Y"].values
fyear = work["fyear"].fillna(0).astype(int).values

# 4. 存檔 (Benchmark 和 Research 使用同一份數據)
data_packet = {
    "X_bench": X_processed,      # 給 Benchmark 用
    "X_research": X_processed,   # 給 Research 用 (兩者完全一樣)
    "feat_names": all_features,
    "y": y, 
    "fyear": fyear
}

joblib.dump(data_packet, os.path.join(OUTPUT_DIR, "data_processed.pkl"))
print(f"EDA Done. Processed data saved for both Benchmark and Research.")

Loading Data...
Features detected: 22
Processing Data (SIC2 Imputation + Log + Winsorize)...
EDA Done. Processed data saved for both Benchmark and Research.
